In [ ]:
import torch
import pandas as pd
from sklearn.metrics import (
    classification_report,
    accuracy_score,
    precision_recall_fscore_support,
    confusion_matrix
)
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification

class PhishingEvaluator:
    def __init__(self, model_path="model"):
        self.tokenizer = XLMRobertaTokenizer.from_pretrained(model_path)
        self.model = XLMRobertaForSequenceClassification.from_pretrained(model_path)
        self.model.eval()

    @torch.no_grad()
    def predict_single(self, text: str):
        inputs = self.tokenizer(
            text,
            truncation=True,
            max_length=256,
            padding=True,
            return_tensors="pt"
        )
        outputs = self.model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        pred = torch.argmax(probs, dim=1).item()
        return pred, float(probs[0][pred])

    @torch.no_grad()
    def predict_batch(self, texts):
        inputs = self.tokenizer(
            texts,
            truncation=True,
            max_length=256,
            padding=True,
            return_tensors="pt"
        )
        outputs = self.model(**inputs)
        probs = torch.softmax(outputs.logits, dim=1)
        preds = torch.argmax(probs, dim=1)
        return preds.cpu().numpy(), probs.cpu().numpy()

    def evaluate_dataset(self, texts, labels):
        preds = []
        for text in texts:
            pred, _ = self.predict_single(text)
            preds.append(pred)

        acc = accuracy_score(labels, preds)
        p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
        report = classification_report(labels, preds, target_names=["Safe", "Phishing"])
        cm = confusion_matrix(labels, preds)

        results = {
            "accuracy": acc,
            "precision": p,
            "recall": r,
            "f1": f1,
            "classification_report": report,
            "confusion_matrix": cm.tolist()
        }
        return results


In [ ]:
import pandas as pd

evaluator = 

df = pd.read_csv("other_dataset.csv")

results = evaluator.evaluate_dataset(df["text"], df["label"])
print(results["classification_report"])
print("Confusion Matrix:", results["confusion_matrix"])


In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

cm = results["confusion_matrix"]

sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Safe", "Phishing"],
            yticklabels=["Safe", "Phishing"])
plt.xlabel("Predicted")
plt.ylabel("Actual")
plt.show()


In [ ]:
import os
import json
import pandas as pd
from sklearn.metrics import (
    accuracy_score,
    precision_recall_fscore_support,
    classification_report,
)
import torch
from transformers import XLMRobertaTokenizer, XLMRobertaForSequenceClassification

# ===========================================
# 1. Paths (UPDATE ONLY THESE TWO)
# ===========================================
MODEL_PATH = "core\\models\\xlm-r-phishing-final"
DATASETS_DIR = "datasets\\evaluation\\english"
OUTPUT_PATH = "datasets\\evaluation\\results_english.json"

# ===========================================
# 2. Load Fine-Tuned Model
# ===========================================
device = "cuda" if torch.cuda.is_available() else "cpu"

model = XLMRobertaForSequenceClassification.from_pretrained(MODEL_PATH)
model.to(device)
model.eval()
tokenizer = XLMRobertaTokenizer.from_pretrained(MODEL_PATH)

@torch.no_grad()
def predict_batch(texts):
    inputs = tokenizer(
        texts,
        truncation=True,
        max_length=256,
        padding=True,
        return_tensors="pt"
    ).to(device)

    outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)
    preds = torch.argmax(probs, dim=1)

    return preds.cpu().numpy(), probs.cpu().numpy()

def evaluate_dataset(df):
    texts = df["text"].astype(str).tolist()
    labels = df["label"].tolist()

    preds, _ = predict_batch(texts)

    acc = accuracy_score(labels, preds)
    p, r, f1, _ = precision_recall_fscore_support(labels, preds, average="binary")
    report = classification_report(labels, preds, target_names=["Safe", "Phishing"], output_dict=True)

    return {
        "accuracy": acc,
        "precision": p,
        "recall": r,
        "f1": f1,
        "detailed_report": report,
        "samples": len(df),
    }

# ===========================================
# 3. Evaluate ALL CSV files in english folder
# ===========================================
results = {}

csv_files = [f for f in os.listdir(DATASETS_DIR) if f.endswith(".csv")]

if not csv_files:
    print("❌ No CSV files found in", DATASETS_DIR)

for filename in csv_files:
    dataset_path = os.path.join(DATASETS_DIR, filename)
    print(f"\n🟦 Evaluating {filename} ...")

    try:
        df = pd.read_csv(dataset_path)

        # Ensure correct columns
        if "text" not in df.columns or "label" not in df.columns:
            print(f"⚠️ Skipping {filename}: missing 'text' or 'label'")
            continue

        metrics = evaluate_dataset(df)
        results[filename] = metrics

        print(f"✔ Done {filename}: F1 = {metrics['f1']:.4f}")

    except Exception as e:
        print(f"❌ Failed to evaluate {filename}: {e}")

# ===========================================
# 4. Save results to JSON
# ===========================================
with open(OUTPUT_PATH, "w") as f:
    json.dump(results, f, indent=4)

print("\n========================================")
print("✅ Evaluation completed!")
print(f"📄 Results saved to: {OUTPUT_PATH}")
print("========================================")


d:\Hackathon\Unifonic\hate & spam detection\fraud-detection\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm



🟦 Evaluating CEAS_08.csv ...
